### Load data

In [ ]:
import pandas as pd

questions = pd.read_csv("data/questions.csv")   # Data with raw parameters info
instructions = pd.read_csv("data/instructions.csv")   # Data in instructions style (instruction converted from parameters)

In [11]:
len(questions), len(instructions)

(8312, 8312)

#### Check if two data are match

In [12]:
# Check if they are match

for i in range(len(instructions)):
    q_exercise_types = questions.iloc[i]["exercise_type"].split(",")
    # print(q_exercise_types)
    [t.lower() in instructions.iloc[i]["Input"].lower() for t in q_exercise_types]
    if all(t.lower() in instructions.iloc[i]["Input"].lower() for t in q_exercise_types):
        # print(True, i)
        pass
    else:
        print(False, i)
    # break

#### Create coherence score (0-4) test dataset

In [ ]:
"""
Define sample function.

Sample two random (params,excercise) instance, use the second exercise as the generated excerise of the first params
The match params are score of coherence.
"""
import numpy as np
import random

# Define function to sample and calculate matches
def sample_and_match(df):
    """
    Sample two (can be the same) row and calculate match params as score
    """
    # With 0.2 probability take the input output from the same sample to have score of 4
    if random.random() < 0.2:
        # Sample the same row
        sampled_rows = df.sample(1)
        # Use the same index
        idx1 = idx2 = sampled_rows.index[0]
    else:
        # Sample 2 rows
        sampled_rows = df.sample(2)
        # Extract the indices of the sampled rows
        idx1, idx2 = sampled_rows.index
        
    # Exclude "content" and "difficulty" column from the comparison
    cols_to_compare = df.columns.difference(['content', 'difficulty'])

    # Count the number of matching columns (excluding "content" and "difficulty")
    match = df.iloc[idx1][cols_to_compare] == df.iloc[idx2][cols_to_compare]
    num_matched_columns = np.sum(match)
    
    # Create a result dictionary
    result = {
        'score': num_matched_columns,
        'index1': idx1,
        'index2': idx2
    } | match.astype(int).to_dict() # Merge two dictionaries
    
    return result

# Example usage
result_df = sample_and_match(questions)
print(result_df)

{'score': 1, 'index1': 6640, 'index2': 7134, 'exercise_type': 0, 'grade_level': 0, 'modifier': 0, 'subject': 1}


In [30]:
questions.iloc[result_df["index1"]], questions.iloc[result_df["index2"]]

(subject                               Mathematics
 exercise_type                    Drawing Exercise
 difficulty                                      D
 grade_level                                     2
 modifier                                        M
 content          Draw a right angle and label it.
 Name: 8253, dtype: object,
 subject                               Mathematics
 exercise_type                    Drawing Exercise
 difficulty                                      D
 grade_level                                     2
 modifier                                        M
 content          Draw a right angle and label it.
 Name: 8253, dtype: object)

In [ ]:
""" 
Generate a coherence score sample pool which should be big enough for later even distribution sampling
"""

# Generate a sample pool of size 20000
results = []
for _ in range(20000):
    result = sample_and_match(questions)
    results.append(result)

# Convert the results to a DataFrame
results_df = pd.DataFrame(results)

# Display the resulting DataFrame
# print(results_df)

# Display the distribution of different values of num_matched_column
distribution = results_df['score'].value_counts().sort_index()

# Display the distribution
print(distribution)

score
0    10506
1     4363
2      981
3      110
4     4040
Name: count, dtype: int64


In [ ]:
"""
Create even distribution coherence score dataset with each score having 100 instances
"""

# Sample 50 values from each category of score
def sample_from_each_category(df, column, sample_size=50):
    # Group by the specified column (score)
    grouped = df.groupby(column)
    
    # Sample up to 'sample_size' rows from each group
    sampled_df = grouped.apply(lambda x: x.sample(min(len(x), sample_size), replace=False)).reset_index(drop=True)
    
    return sampled_df

# Apply the function to sample 50 rows from each category of score (0-4)
sampled_results_df = sample_from_each_category(results_df, 'score', sample_size=100)

# Display the new DataFrame with the sampled results
print(sampled_results_df['score'].value_counts().sort_index())

score
0    100
1    100
2    100
3    100
4    100
Name: count, dtype: int64


In [ ]:
# sampled_results_df.to_csv("data/coherence_eval.csv", index=False)

### Evaluate the method `coherence evaluation by llm`

In [ ]:
sampled_results_df = pd.read_csv("data/coherence_eval.csv")

In [15]:
# Sample "Input" based on row1.index and "Output" based on row2.index
sampled_results_df['Input'] = sampled_results_df['index1'].apply(lambda idx: instructions.loc[idx, 'Input'])
sampled_results_df['Output'] = sampled_results_df['index2'].apply(lambda idx: instructions.loc[idx, 'Output'])
sampled_results_df['OriInput'] = sampled_results_df['index2'].apply(lambda idx: instructions.loc[idx, 'Input'])
sampled_results_df['OriOutput'] = sampled_results_df['index1'].apply(lambda idx: instructions.loc[idx, 'Output'])

In [16]:
shuffle_data = sampled_results_df.sample(frac=1, ignore_index=True)

In [ ]:
"""
Define llm calling function
"""

from openai import OpenAI
from dotenv import load_dotenv

# Set up your OpenAI API key
load_dotenv() 
client = OpenAI()

# Text generation using GPT-4
def generate_text(sys_prompt, user_prompt):
    try:
        response = client.chat.completions.create(
            model="gpt-4o",  # Use "gpt-4-turbo" for a faster, cheaper alternative
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=150,  # Limit the number of tokens in the output
            temperature=0.7,  # Adjust the creativity level of the model
            n=1,  # Number of completions to generate
            stop=None  # Stop generation based on special characters or tokens
        )
        return response.choices[0].message.content
    
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# # Example usage
# prompt = "Write a short story about a robot learning to play the piano."
# generated_text = generate_text(prompt)
# print(generated_text)

In [ ]:
"""
Define coherence score function
"""
import re

system_prompt = """
Role:
You are an exercise generation evaluator. 

Task:
Given an query-answer pair, where the query is a demand and the answer is the generated exercise,
you will score the coherence of the excercise from 4 perspectives:
- Subject: score 1 if the subject is correct else 0.
- Exercise type: score 1 if the exercise type is correct else 0.
- Grade level: score 1 if the grade level is close to the given according to 1-12 Amecian grade system else 0.
- Modification: score 1 if the specific details matches the requirement else 0.
Don't be too strict in each perspective, you can give them score as long as it is fulfilled in a cetain level. Your final score is the sum of all these four scores.

Return:
Provide your answer in the following format:
{"Subject score": x, "Exercise type score": x, "Grade level score": x, "Modification score": x, "Total score": x}"""

def parse_response(response):
    response = re.findall(r"{[\s\S]+}", response)[0]
    return eval(response)

result_list = []

def coherence_eval(x):
    input, output = x['Input'], x['Output']
    user_prompt = str({"Query": input, "Answer": output})
    try:
        response = generate_text(system_prompt, user_prompt)
        result = parse_response(response)
    except Exception as e:
        print(e)
        print(response)
        return {'Subject score': None, 'Exercise type score': None, 'Grade level score': None, 'Modification score': None, 'Total score': None}

    return result

In [ ]:
sampled_results_df = shuffle_data[:100]

In [ ]:
# """
# Predict coherence scores (by for foop)
# """

# predict_scores = {}
# for index, row in sampled_results_df.iterrows():
#     input, output = row['Input'], row['Output']
#     user_prompt = f"Query: \"{input}\"\n\nAnswer: \"{output}\""
#     response = generate_text(system_prompt, user_prompt)
#     result = parse_response(response)
#     predict_scores[index] = result["Total score"]
#     break

In [ ]:
"""
Predict coherence scores
"""

sampled_results_df[['Subject score', 'Exercise type score', 'Grade level score', 'Modification score', 'Total score']] = \
    sampled_results_df.apply(coherence_eval, axis=1, result_type='expand')

/tmp/ipykernel_17520/3161120063.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sampled_results_df[['Subject score', 'Exercise type score', 'Grade level score', 'Modification score', 'Total score']] = \


In [33]:
sampled_results_df.to_csv('data/coherence_eval_result_2.csv', index=False)

In [ ]:
"""
Reload coherence score prediction and calculate accuracy
"""

import numpy as np
import pandas as pd

data = pd.read_csv('data/coherence_eval_result_2.csv')

subject_correct = sum(data['subject'] == data['Subject score']) / len(data)
type_correct = sum(data['exercise_type'] == data['Exercise type score']) / len(data)
grade_correct = sum(data['grade_level'] == data['Grade level score']) / len(data)
modifier_correct = sum(data['modifier'] == data['Modification score']) / len(data)
mAA = np.mean([subject_correct, type_correct, grade_correct, modifier_correct])

print(f"Evaluate by llm accuracy:\nSubject: {subject_correct},\nExercise type: {type_correct},\nGrade level: {grade_correct},\
      \nModifier: {modifier_correct},\nMean Average Accuracy: {mAA}")

In [ ]:
# 0: (0.73, 0.79, 0.78, 0.83, 0.7825)
# 2: (0.76, 0.77, 0.81, 0.85, 0.7975)